In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('../data/scrap-apify/data_instagram.csv')
data.head()

,username,fullName,biography,followersCount,followsCount,postsCount,profilePicUrl,private,verified,id
0,arofahapril97,fah_april08,NaN,226.0,265.0,0.0,https://scontent-atl3-1.cdninstagram.com/v/t51...,True,False,2.300268e+09
1,hestyntln,hestyntln,......,1131.0,1986.0,1.0,https://instagram.frec6-1.fna.fbcdn.net/v/t51....,True,False,3.891035e+10
2,aaayyuu29,ayu melatiee,@arlano0o,1026.0,505.0,0.0,https://instagram.feoh9-2.fna.fbcdn.net/v/t51....,True,False,5.984734e+10
3,sfaaashc_,aerashaac,i67\nUnCambrige,518.0,89.0,1.0,https://scontent-atl3-1.cdninstagram.com/v/t51...,True,False,6.055380e+10
4,wiwik_mugirahayu,Wiwik Rahayu,Kosong,268.0,253.0,0.0,https://scontent-sea5-1.cdninstagram.com/v/t51...,True,False,6.306755e+10


In [3]:
# data shape
print(f"Data shape: {data.shape}")

Data shape: (1000, 10)


In [4]:
# data.info()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   username        1000 non-null   object 
 1   fullName        782 non-null    object 
 2   biography       519 non-null    object 
 3   followersCount  994 non-null    float64
 4   followsCount    994 non-null    float64
 5   postsCount      994 non-null    float64
 6   profilePicUrl   995 non-null    object 
 7   private         995 non-null    object 
 8   verified        995 non-null    object 
 9   id              995 non-null    float64
dtypes: float64(4), object(6)
memory usage: 78.2+ KB


In [5]:
# missing & duplicated values
print(f"Missing values:\n{data.isnull().sum()}")
print(f"Duplicated values: {data.duplicated().sum()}")

Missing values:
username            0
fullName          218
biography         481
followersCount      6
followsCount        6
postsCount          6
profilePicUrl       5
private             5
verified            5
id                  5
dtype: int64
Duplicated values: 0


In [6]:
data[data['profilePicUrl'].isna()]

,username,fullName,biography,followersCount,followsCount,postsCount,profilePicUrl,private,verified,id
180,laure.ngootee1357068,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,ismayaputrilestari100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
802,cindy_okt4a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
813,savvygivaways2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
985,dstriani012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
data[data['id'].duplicated()]

,username,fullName,biography,followersCount,followsCount,postsCount,profilePicUrl,private,verified,id
181,ismayaputrilestari100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
802,cindy_okt4a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
813,savvygivaways2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
985,dstriani012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
data['username'].duplicated().sum()

0

# SIMULASI PROCESSING.PY

In [9]:
data.drop(columns=['id'], inplace=True)
data.head()

,username,fullName,biography,followersCount,followsCount,postsCount,profilePicUrl,private,verified
0,arofahapril97,fah_april08,NaN,226.0,265.0,0.0,https://scontent-atl3-1.cdninstagram.com/v/t51...,True,False
1,hestyntln,hestyntln,......,1131.0,1986.0,1.0,https://instagram.frec6-1.fna.fbcdn.net/v/t51....,True,False
2,aaayyuu29,ayu melatiee,@arlano0o,1026.0,505.0,0.0,https://instagram.feoh9-2.fna.fbcdn.net/v/t51....,True,False
3,sfaaashc_,aerashaac,i67\nUnCambrige,518.0,89.0,1.0,https://scontent-atl3-1.cdninstagram.com/v/t51...,True,False
4,wiwik_mugirahayu,Wiwik Rahayu,Kosong,268.0,253.0,0.0,https://scontent-sea5-1.cdninstagram.com/v/t51...,True,False


In [10]:
def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename columns sesuai standar project
    """
    return df.rename(columns={
        "fullName": "fullname",
        "biography": "bio",
        "followersCount": "#followers",
        "followsCount": "#follows",
        "postsCount": "#posts",
        "profilePicUrl": "profile_pic"
    })


def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Handle missing values sesuai rule:
    - username: drop
    - fullname: '-'
    - bio: '-'
    - #followers, #follows, #posts, private: drop rows
    - profile_pic: convert ke boolean (True jika ada, False jika tidak)
    - verified: isi dengan modus
    """

    
    df = df.dropna(subset=["username"])

    
    df["fullname"] = df["fullname"].fillna("-")
    df["bio"] = df["bio"].fillna("-")

    
    df["profile_pic"] = df["profile_pic"].notna()

    
    df = df.dropna(subset=[
        "#followers", "#follows", "#posts", "private"
    ])

    
    if df["verified"].isnull().sum() > 0:
        mode_val = df["verified"].mode()[0]
        df["verified"] = df["verified"].fillna(mode_val)

    return df


def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full preprocessing pipeline
    """
    df = rename_columns(df)
    df = handle_missing_values(df)

    return df

In [11]:
data_processed = preprocess_data(data)
data_processed.head()

,username,fullname,bio,#followers,#follows,#posts,profile_pic,private,verified
0,arofahapril97,fah_april08,-,226.0,265.0,0.0,True,True,False
1,hestyntln,hestyntln,......,1131.0,1986.0,1.0,True,True,False
2,aaayyuu29,ayu melatiee,@arlano0o,1026.0,505.0,0.0,True,True,False
3,sfaaashc_,aerashaac,i67\nUnCambrige,518.0,89.0,1.0,True,True,False
4,wiwik_mugirahayu,Wiwik Rahayu,Kosong,268.0,253.0,0.0,True,True,False


In [12]:
# check missing, duplicated
print(f"Missing values:\n{data_processed.isnull().sum()}")
print(f"Duplicated values: {data_processed.duplicated().sum()}")

Missing values:
username       0
fullname       0
bio            0
#followers     0
#follows       0
#posts         0
profile_pic    0
private        0
verified       0
dtype: int64
Duplicated values: 0


# SIMULASI feature_engineering.py

In [13]:
# feature_engineering.py
import numpy as np
import re


def _count_numbers(text: str) -> int:
    return len(re.findall(r"\d", text))


def add_username_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """
    nums/length_username
    """
    df["nums/length_username"] = df["username"].apply(
        lambda x: _count_numbers(x) / len(x) if len(x) > 0 else 0
    )
    return df


def add_fullname_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    fullname_words & nums/length_fullname
    """
    df["fullname_words"] = df["fullname"].apply(lambda x: len(str(x).split()))
    df["nums/length_fullname"] = df["fullname"].apply(
        lambda x: _count_numbers(x) / len(x) if len(x) > 0 else 0
    )
    return df


def add_name_comparison(df: pd.DataFrame) -> pd.DataFrame:
    """
    fullname == username
    """
    df["fullname==username"] = df["fullname"] == df["username"]
    return df


def add_description_length(df: pd.DataFrame) -> pd.DataFrame:
    """
    description_length dari bio
    """
    df["description_length"] = df["bio"].apply(lambda x: len(str(x)))
    return df


def reorder_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Urutkan kolom sesuai requirement
    """
    final_cols = [
        "profile_pic",
        "nums/length_username",
        "fullname_words",
        "nums/length_fullname",
        "fullname==username",
        "description_length",
        "verified",
        "private",
        "#posts",
        "#followers",
        "#follows"
    ]

    return df[final_cols]


def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full feature engineering pipeline
    """
    df = add_username_ratio(df)
    df = add_fullname_features(df)
    df = add_name_comparison(df)
    df = add_description_length(df)

    df = reorder_columns(df)

    return df

In [14]:
data_processed = feature_engineering(data_processed)
data_processed.head()

,profile_pic,nums/length_username,fullname_words,nums/length_fullname,fullname==username,description_length,verified,private,#posts,#followers,#follows
0,True,0.153846,1,0.181818,False,1,False,True,0.0,226.0,265.0
1,True,0.000000,1,0.000000,True,6,False,True,1.0,1131.0,1986.0
2,True,0.222222,2,0.000000,False,9,False,True,0.0,1026.0,505.0
3,True,0.000000,1,0.000000,False,14,False,True,1.0,518.0,89.0
4,True,0.000000,2,0.000000,False,6,False,True,0.0,268.0,253.0


# SIMULASI MANGGIL preprocessing.py dan feature_engineering.py


#### note: preprocessing.py dan feature_engineering.py sudah ada di ../src

In [16]:
#setup
import sys
import os

sys.path.append(os.path.abspath('../src'))

from sklearn.pipeline import Pipeline
from pipeline import PreprocessingTransformer, FeatureEngineeringTransformer

In [17]:
pipeline = Pipeline([
    ("preprocessing", PreprocessingTransformer()),
    ("feature_engineering", FeatureEngineeringTransformer())
])

df_processed = pipeline.fit_transform(data)
df_processed.head()

,profile_pic,nums/length_username,fullname_words,nums/length_fullname,fullname==username,description_length,verified,private,#posts,#followers,#follows
0,True,0.153846,1,0.181818,False,1,False,True,0.0,226.0,265.0
1,True,0.000000,1,0.000000,True,6,False,True,1.0,1131.0,1986.0
2,True,0.222222,2,0.000000,False,9,False,True,0.0,1026.0,505.0
3,True,0.000000,1,0.000000,False,14,False,True,1.0,518.0,89.0
4,True,0.000000,2,0.000000,False,6,False,True,0.0,268.0,253.0
